**Author**: Felipe Matheus  
**Start Date**: --/05/2026  
**End Date**: --/--/2026  

**Objective**

Evaluate the cold-drawing UTS surrogate in the **multi-step rollout** regime: for each wire in the held-out set, predict the full sequence of post-pass states starting from the real $\bm s_0$ and feeding each predicted state back as input to the next pass. Compare against the wire's true measured trajectory.

This regime mirrors how the model-based controller (MBC) actually uses the surrogate at deployment, and exposes **error compounding** that the one-step metric of the companion notebook cannot detect.

**Prerequisites**

Run `cold_drawing_uts_one_step.ipynb` first. This notebook loads the trained predictors and artifacts saved by it; nothing is retrained here.

**Pipeline**

1. Setup and load artifacts from the one-step notebook.
2. Build a recalibrated, packaged predict function.
3. Run rollout on every wire in the dataset (or a held-out fold).
4. Aggregate metrics: one-step vs final-state, pass-by-pass curve, calibration on the final state.
5. Compounding ratio: how much faster does error grow than the independent-noise lower bound?
6. Single-wire rollout visualisation.

**Caveat on this notebook's split**

For a fully honest multi-step evaluation, the wires used here must be **held out** from Model A and Model B training. AutoGluon's GroupKFold internal bagging gives us out-of-fold predictions on every training wire, which is what we use as a first-order honest estimate. A truly external test set (separate wires never seen by any fold) would be ideal once more data is available.

# 1. Setup and load artifacts

In [2]:
import os
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from autogluon.tabular import TabularPredictor
from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.feature_engineering.CDHelper import CDHelper
from src.modeling.Modeling import Modeling
from src.modeling.Evaluation import Evaluation
from config.Variables import Variables

%load_ext autoreload
%autoreload 2

In [3]:
proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
varv = Variables()

In [ ]:
# Same paths and file names as the one-step notebook.
PATH_DATA_RAW = '../../data/raw'
PATH_MODELS = '../../models'
RAW_DATASET_FILE_NAME = 'dataset_cold_drawing_uts.csv'
PROCESS = 'cold_drawing_uts'

# ---- Load artifacts ----
with open(os.path.join(PATH_MODELS, PROCESS, 'artifacts.pkl'), 'rb') as f:
    art = pickle.load(f)

predictor_a = TabularPredictor.load(os.path.join(PATH_MODELS, PROCESS, 'model_a'))
predictor_b = TabularPredictor.load(os.path.join(PATH_MODELS, PROCESS, 'model_b'))

FEATURES = art['features']
TARGET = art['target']
GROUP_COL = art['group_col']
RECURSIVE_FEATURE = art['recursive_feature']
SORT_COLUMN = art['sort_column']

print('Artifacts loaded.')
print(f'  features        = {FEATURES}')
print(f'  target          = {TARGET}')
print(f'  group_col       = {GROUP_COL}')
print(f'  recursive_feature = {RECURSIVE_FEATURE}')
print(f'  recalibration_c = {art["recalibration_c"]:.4f}')

This means that the predictor was fit in an AutoGluon version `<=0.3.1`.


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\fmfoa\\Projects\\uncertainty-aware-predictors\\models\\cold_drawing_uts\\model_a\\predictor.pkl'

# 2. Build the packaged predict function

The rollout needs a closure that takes a single-row (or small-batch) DataFrame of features and returns `mu`, `sigma_total` already in the right scale and with the scalar recalibration applied. We build it once here and reuse it for the rollout.

In [ ]:
def predict_fn(X):
    return modl.predict_with_uncertainty(
        X=X,
        predictor_a=predictor_a,
        predictor_b=predictor_b,
        weights=art['weights'],
        model_names=art['base_model_names'],
        features=FEATURES,
        variance_floor=art['variance_floor'],
        recalibration_c=art['recalibration_c'],
        use_weights=art['use_weighted_variance'],
    )

# Quick sanity check on one row.
print('Sanity check (one prediction):')
# We do not have df loaded yet; load it next.


# 3. Load and prepare the dataset

Same preprocessing as the one-step notebook — wire-level grouping plus the `initial_tensile_strength` column derived from the previous pass.

In [ ]:
path = os.path.join(PATH_DATA_RAW, RAW_DATASET_FILE_NAME)
df_raw = pd.read_csv(path).dropna()
df_raw.loc[df_raw['purity'] == '-', 'purity'] = 99.9
df_raw.purity = df_raw.purity.astype(float)

df_grouped = CDHelper.set_group_experiment(df=df_raw)
df_grouped_fixed, _ = CDHelper.check_n_pass(df_grouped, correct=True)
df_full = CDHelper.add_initial_tensile_strength(df_grouped_fixed)
df = df_full[df_full.sanity_check_ok == True].copy()
df = df[FEATURES + [TARGET, GROUP_COL]].reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'Number of wires: {df[GROUP_COL].nunique()}')
df.head()

# 4. Run the rollout on every wire

For each wire, sort by `pass_number`, take the real $\bm s_0$, predict pass 1, feed that prediction back as input to pass 2, and so on until the last pass. Compare to the wire's true measured trajectory.

In [ ]:
rollout = evla.rollout_all_groups(
    df=df,
    group_col=GROUP_COL,
    predict_fn=predict_fn,
    features=FEATURES,
    target=TARGET,
    recursive_feature=RECURSIVE_FEATURE,
    sort_column=SORT_COLUMN,
)
print(f'rollout rows: {len(rollout)}')
print(f'wires processed: {rollout.group_id.nunique()}')
rollout.head(10)

# 5. Aggregate metrics

Three complementary views of the rollout result.

**Final-state metrics** — the model's behaviour on the last pass of each wire. This is the metric that most directly matches what the MBC sees: the predicted final UTS of a candidate route vs the true final UTS.

**Pass-by-pass error curve** — average error as a function of pass index. Reveals compounding: if the curve rises steeply, errors are accumulating fast.

**Final-state coverage** — does the predictive interval at the final pass contain the true value at the nominal frequency?

In [ ]:
# Final-state metrics: only the last pass of each wire.
final_metrics = evla.final_state_metrics(rollout, group_col='group_id')
print('=== Final-state metrics (last pass of each wire) ===')
print(f'  RMSE = {final_metrics["rmse"]:.4f}')
print(f'  MAE  = {final_metrics["mae"]:.4f}')
print(f'  MAPE = {final_metrics["mape"]:.4f} %')
print(f'  R^2  = {final_metrics["r2"]:.4f}')
print('  Coverage at final pass:')
for a, cov in final_metrics['coverage'].items():
    print(f'    alpha={a:.2f}  empirical={cov:.3f}  gap={cov - a:+.3f}')

In [ ]:
# Pass-by-pass curve
curve = evla.pass_by_pass_curve(rollout)
print('=== Pass-by-pass rollout curve ===')
print(curve)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(curve['pass_index'], curve['mean_abs_error'], 'o-', label='mean |error|')
ax[0].plot(curve['pass_index'], curve['median_abs_error'], 's--', alpha=0.7, label='median |error|')
ax[0].plot(curve['pass_index'], curve['mean_sigma'], '^:', alpha=0.7, label='mean predicted sigma')
ax[0].set_xlabel('pass index (0 = first pass)')
ax[0].set_ylabel('MPa')
ax[0].set_title('Compounding curve')
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].plot(curve['pass_index'], curve['coverage_90'], 'o-', color='C2')
ax[1].axhline(0.9, color='k', linestyle='--', alpha=0.5, label='nominal alpha=0.9')
ax[1].set_xlabel('pass index')
ax[1].set_ylabel('empirical coverage')
ax[1].set_title('Per-pass coverage at alpha=0.9')
ax[1].set_ylim(0, 1.05)
ax[1].legend()
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 6. Compounding ratio

How much faster does the final-state RMSE grow than the independent-noise lower bound $\sqrt{n} \times \text{RMSE}_{\text{1-step}}$? A value near 1 means errors are roughly independent across passes (best realistic case). A value near $\sqrt{n}$ means errors are fully correlated and compound aggressively.

We use the wire's typical pass count as $n$.

In [ ]:
one_step_metrics = art['one_step_metrics']
n_passes_typical = int(df.groupby(GROUP_COL).size().median())
print(f'Typical (median) number of passes per wire: {n_passes_typical}')

ratio = evla.compounding_ratio(
    rollout_metrics=final_metrics,
    one_step_metrics=one_step_metrics,
    n_passes_typical=n_passes_typical,
)
print(f'\nOne-step RMSE         = {one_step_metrics["rmse"]:.4f}')
print(f'Final-state RMSE      = {final_metrics["rmse"]:.4f}')
print(f'Compounding ratio     = {ratio:.3f}')
print(f'(Interpretation: ratio ~1 means independent errors; ratio >> 1 means correlated compounding.)')

# 7. Visualise one wire's trajectory

Pick a representative wire and plot the predicted vs true UTS pass by pass, with the predictive interval as a shaded band.

In [ ]:
# Pick the wire whose final-state error is closest to the median.
last = (
    rollout.sort_values(['group_id', 'pass_index'])
           .groupby('group_id').tail(1)
)
median_err = last['abs_error'].median()
target_gid = last.iloc[(last['abs_error'] - median_err).abs().argmin()]['group_id']
print(f'Selected wire: {target_gid}  (median final-state error)')

wire = rollout[rollout.group_id == target_gid].sort_values('pass_index')

fig, ax = plt.subplots(figsize=(9, 4))
passes = wire['pass_number'].values
mu = wire['mu_pred'].values
sigma = wire['sigma_pred'].values
y_true = wire['y_true'].values

ax.plot(passes, y_true, 'ko-', label='true UTS', zorder=3)
ax.plot(passes, mu, 'C0o--', label='predicted mu (rollout)')
ax.fill_between(passes, mu - 1.645 * sigma, mu + 1.645 * sigma,
                alpha=0.3, color='C0', label='90% predictive interval')
ax.set_xlabel('pass number')
ax.set_ylabel('UTS (MPa)')
ax.set_title(f'Rollout trajectory — wire {target_gid}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 8. Acceptance check

Project criteria for cold-drawing UTS surrogate accepting multi-step use:
- Compounding ratio reasonably close to 1 (target: $\leq 1.5$).
- Final-state coverage at $\alpha = 0.9$ within $\pm 5$ percentage points of nominal.
- Pass-by-pass error curve does not explode (i.e., final-pass mean error $\leq$ first-pass mean error $\times \sqrt{n}$).

If any of these fail, the surrogate is fine for one-step use but unsafe for the MBC. Mitigations to try (in order):
1. Apply input augmentation (errors-in-variables) at training time on the recursive feature. See v4 spec §4.5.
2. Investigate systematic bias drift in the pass-by-pass curve.
3. Collect more wire-level data.

In [ ]:
checks = {
    'compounding_ratio_leq_1.5': ratio <= 1.5,
    'final_coverage_within_5pp_of_0.9': abs(final_metrics['coverage'][0.9] - 0.9) <= 0.05,
    'final_pass_error_reasonable': (
        curve['mean_abs_error'].iloc[-1]
        <= curve['mean_abs_error'].iloc[0] * np.sqrt(n_passes_typical)
    ),
}
print('=== Acceptance checks ===')
for k, v in checks.items():
    status = '✓ PASS' if v else '✗ FAIL'
    print(f'  {status}  {k}')